# Prompt-based image segmentation with MobileSAM and OpenVINO

[MobileSAM](https://github.com/ChaoningZhang/MobileSAM) replaces the original Segment Anything image encoder with a lightweight <spell>TinyViT</spell> encoder while retaining prompt-based mask decoding. This tutorial converts the original PyTorch checkpoint **directly to OpenVINO IR without an ONNX intermediate**, validates numerical accuracy, and demonstrates point- and box-prompt inference.

Learn more in the [MobileSAM paper](https://arxiv.org/abs/2306.14289) and the [official source repository](https://github.com/ChaoningZhang/MobileSAM). The model and source code are distributed under the [Apache License 2.0](https://github.com/ChaoningZhang/MobileSAM/blob/master/LICENSE).

#### Table of contents:

- [Install MobileSAM dependencies](#Install-MobileSAM-dependencies)
- [Import libraries and notebook utilities](#Import-libraries-and-notebook-utilities)
- [Configure model paths and runtime device](#Configure-model-paths-and-runtime-device)
- [Download MobileSAM source and checkpoint](#Download-MobileSAM-source-and-checkpoint)
- [Load the original MobileSAM model](#Load-the-original-MobileSAM-model)
- [Prepare example inputs](#Prepare-example-inputs)
- [Wrap MobileSAM components for conversion](#Wrap-MobileSAM-components-for-conversion)
- [Convert MobileSAM to OpenVINO IR](#Convert-MobileSAM-to-OpenVINO-IR)
- [Validate OpenVINO model accuracy](#Validate-OpenVINO-model-accuracy)
- [Compile the OpenVINO models](#Compile-the-OpenVINO-models)
- [Implement OpenVINO MobileSAM inference](#Implement-OpenVINO-MobileSAM-inference)
- [Run inference on a sample image](#Run-inference-on-a-sample-image)
- [Create an interactive segmentation demo](#Create-an-interactive-segmentation-demo)
- [Measure model size and performance](#Measure-model-size-and-performance)
- [Verify notebook reproducibility](#Verify-notebook-reproducibility)
    - [Optional cleanup](#Optional-cleanup)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/mobilesam-segmentation/mobilesam-segmentation.ipynb" />

## Install MobileSAM dependencies
[back to top ⬆️](#Table-of-contents:)

Install only the packages required by this tutorial. The extra PyTorch index provides CPU wheels on systems without an existing PyTorch installation.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

UTILS_PATH = Path("notebook_utils.py")
if not UTILS_PATH.exists():
    urlretrieve(
        "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
        UTILS_PATH,
    )

from notebook_utils import pip_install

print("Installing MobileSAM dependencies. The first run can take several minutes...")
pip_install(
    "openvino>=2025.0",
    "torch>=2.1",
    "torchvision>=0.16",
    "timm>=0.9.2",
    "numpy<2.3",
    "pillow>=9.4",
    "matplotlib>=3.7",
    "opencv-python>=4.8",
    "ipywidgets>=8.0",
    "gradio>=4.44,<6",
    "huggingface-hub<1.0",
    "requests",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)
print("Dependency installation completed.")

## Import libraries and notebook utilities
[back to top ⬆️](#Table-of-contents:)


In [2]:
import sys
import time
from typing import Optional

import cv2
import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import openvino as ov
import torch
from PIL import Image

from notebook_utils import collect_telemetry, device_widget, download_file

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
collect_telemetry("mobilesam-segmentation.ipynb")

## Configure model paths and runtime device
[back to top ⬆️](#Table-of-contents:)

The encoder is run once per image and its embedding is reused for each prompt. Select any device available to OpenVINO; `AUTO` is also supported.

In [ ]:
MODEL_DIR = Path("model")
DATA_DIR = Path("data")
SOURCE_DIR = Path("mobile-sam")
CHECKPOINT_PATH = MODEL_DIR / "mobile_sam.pt"
ENCODER_PATH = MODEL_DIR / "mobile_sam_image_encoder.xml"
DECODER_PATH = MODEL_DIR / "mobile_sam_prompt_decoder.xml"
SAMPLE_PATH = DATA_DIR / "coco_bike.jpg"

MODEL_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

device = device_widget(default="CPU")
device

## Download MobileSAM source and checkpoint
[back to top ⬆️](#Table-of-contents:)

For reproducibility, this notebook downloads the official repository and `vit_t` checkpoint at commit `f706ad9c4eb7f219c00d9050e46328518ffb65d2`. Neither asset is stored in this repository.

In [ ]:
import zipfile

MOBILESAM_REVISION = "f706ad9c4eb7f219c00d9050e46328518ffb65d2"
SOURCE_URL = f"https://github.com/ChaoningZhang/MobileSAM/archive/{MOBILESAM_REVISION}.zip"
CHECKPOINT_URL = f"https://raw.githubusercontent.com/ChaoningZhang/MobileSAM/{MOBILESAM_REVISION}/weights/mobile_sam.pt"

if not SOURCE_DIR.exists():
    source_archive = download_file(url=SOURCE_URL, filename=f"MobileSAM-{MOBILESAM_REVISION}.zip")
    with zipfile.ZipFile(source_archive) as archive:
        archive.extractall(".")
    source_archive.unlink()
    Path(f"MobileSAM-{MOBILESAM_REVISION}").rename(SOURCE_DIR)

if not CHECKPOINT_PATH.exists():
    download_file(url=CHECKPOINT_URL, filename=CHECKPOINT_PATH.name, directory=MODEL_DIR)

source_path = str(SOURCE_DIR.resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)

assert CHECKPOINT_PATH.stat().st_size > 1_000_000
print(f"MobileSAM source: {SOURCE_DIR.resolve()}")
print(f"Checkpoint: {CHECKPOINT_PATH.resolve()}")

## Load the original MobileSAM model
[back to top ⬆️](#Table-of-contents:)

The checkpoint is loaded on CPU and gradients are disabled because the model is used only for conversion and validation. MobileSAM is split at the image embedding: a comparatively expensive image encoder and a lightweight prompt decoder.

In [ ]:
from mobile_sam import sam_model_registry
from mobile_sam.utils.onnx import SamOnnxModel

sam = sam_model_registry["vit_t"](checkpoint=str(CHECKPOINT_PATH))
sam.eval()
sam.requires_grad_(False)

parameter_count = sum(parameter.numel() for parameter in sam.parameters())
print(f"Model: {sam.__class__.__name__}")
print(f"Parameters: {parameter_count / 1e6:.2f} million")
print(f"Encoder input size: {sam.image_encoder.img_size} x {sam.image_encoder.img_size}")
print(f"Image embedding size: {sam.prompt_encoder.image_embedding_size}")

## Prepare example inputs
[back to top ⬆️](#Table-of-contents:)

For consistency with other OpenVINO Notebooks vision examples, this notebook reuses the repository's `coco_bike.jpg` sample. It is downloaded from OpenVINO Notebooks storage and is not committed to this notebook directory.

In [ ]:
SAMPLE_URL = "https://storage.openvinotoolkit.org/repositories/openvino_notebooks/data/data/image/coco_bike.jpg"
if not SAMPLE_PATH.exists():
    download_file(url=SAMPLE_URL, filename=SAMPLE_PATH.name, directory=DATA_DIR)

image = np.asarray(Image.open(SAMPLE_PATH).convert("RGB"))
original_size = image.shape[:2]

from mobile_sam.utils.transforms import ResizeLongestSide

transform = ResizeLongestSide(sam.image_encoder.img_size)
resized_image = transform.apply_image(image)
input_size = resized_image.shape[:2]
image_tensor = torch.as_tensor(resized_image).permute(2, 0, 1).contiguous()[None].float()
encoder_input = sam.preprocess(image_tensor)

point_coords = torch.tensor([[[image.shape[1] * 0.69, image.shape[0] * 0.50]]], dtype=torch.float32)
point_coords = torch.as_tensor(transform.apply_coords(point_coords.numpy(), original_size), dtype=torch.float32)
point_labels = torch.tensor([[1.0]], dtype=torch.float32)
mask_input = torch.zeros((1, 1, 256, 256), dtype=torch.float32)
has_mask_input = torch.zeros(1, dtype=torch.float32)
original_size_tensor = torch.tensor(original_size, dtype=torch.float32)

with torch.no_grad():
    image_embeddings = sam.image_encoder(encoder_input)

print(f"Image: {image.shape}")
print(f"Encoder input: {tuple(encoder_input.shape)}")
print(f"Image embedding: {tuple(image_embeddings.shape)}")

## Wrap MobileSAM components for conversion
[back to top ⬆️](#Table-of-contents:)

`SamOnnxModel` is a tensor-only export wrapper from the official MobileSAM source. Despite its historical name, it is a regular `torch.nn.Module` and is passed directly to `ov.convert_model`; this workflow never creates or reads an ONNX model. It includes prompt encoding, mask decoding, and resizing masks back to the original image size.

In [ ]:
class ImageEncoderWrapper(torch.nn.Module):
    """Expose the MobileSAM image encoder through a tensor-only forward method."""

    def __init__(self, model: torch.nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(self, input_image: torch.Tensor) -> torch.Tensor:
        """Return image embeddings for a preprocessed BCHW tensor."""
        return self.model(input_image)


encoder_wrapper = ImageEncoderWrapper(sam.image_encoder).eval()
decoder_wrapper = SamOnnxModel(model=sam, return_single_mask=False).eval()
decoder_inputs = {
    "image_embeddings": image_embeddings,
    "point_coords": point_coords,
    "point_labels": point_labels,
    "mask_input": mask_input,
    "has_mask_input": has_mask_input,
    "orig_im_size": original_size_tensor,
}

with torch.no_grad():
    reference_masks, reference_scores, reference_low_res = decoder_wrapper(**decoder_inputs)

print(f"Masks: {tuple(reference_masks.shape)}")
print(f"IoU predictions: {tuple(reference_scores.shape)}")
print(f"Low-resolution masks: {tuple(reference_low_res.shape)}")

## Convert MobileSAM to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

The decoder's number of prompts is made dynamic, allowing one converted model to accept points, boxes (encoded as two corner prompts), or combinations of both. Existing IR files are reused to keep subsequent runs fast.

In [ ]:
if not ENCODER_PATH.exists():
    ov_encoder = ov.convert_model(encoder_wrapper, example_input=encoder_input)
    ov_encoder.inputs[0].get_tensor().set_names({"input_image"})
    ov_encoder.outputs[0].get_tensor().set_names({"image_embeddings"})
    ov.save_model(ov_encoder, ENCODER_PATH, compress_to_fp16=True)
    print(f"Saved {ENCODER_PATH}")
else:
    ov_encoder = ov.Core().read_model(ENCODER_PATH)
    print(f"Reusing {ENCODER_PATH}")

if not DECODER_PATH.exists():
    ov_decoder = ov.convert_model(decoder_wrapper, example_input=decoder_inputs)
    decoder_input_names = list(decoder_inputs)
    decoder_output_names = ["masks", "iou_predictions", "low_res_masks"]
    for model_input, name in zip(ov_decoder.inputs, decoder_input_names):
        model_input.get_tensor().set_names({name})
    for model_output, name in zip(ov_decoder.outputs, decoder_output_names):
        model_output.get_tensor().set_names({name})

    embed_channels = sam.prompt_encoder.embed_dim
    embed_height, embed_width = sam.prompt_encoder.image_embedding_size
    ov_decoder.reshape(
        {
            "image_embeddings": [1, embed_channels, embed_height, embed_width],
            "point_coords": ov.PartialShape([1, -1, 2]),
            "point_labels": ov.PartialShape([1, -1]),
            "mask_input": [1, 1, 4 * embed_height, 4 * embed_width],
            "has_mask_input": [1],
            "orig_im_size": [2],
        }
    )
    ov.save_model(ov_decoder, DECODER_PATH, compress_to_fp16=True)
    print(f"Saved {DECODER_PATH}")
else:
    ov_decoder = ov.Core().read_model(DECODER_PATH)
    print(f"Reusing {DECODER_PATH}")

## Validate OpenVINO model accuracy
[back to top ⬆️](#Table-of-contents:)

Conversion is checked component-by-component using identical inputs. The saved models use FP16 weight compression, so small numerical differences are expected. Final masks are also compared with intersection over union (IoU), which directly measures segmentation agreement.

In [ ]:
def report_difference(name: str, reference: np.ndarray, actual: np.ndarray) -> None:
    """Print maximum and mean absolute differences between two tensors."""
    difference = np.abs(reference.astype(np.float32) - actual.astype(np.float32))
    print(f"{name:22s} max abs diff: {difference.max():.6f}; mean abs diff: {difference.mean():.6f}")


validation_core = ov.Core()
validation_encoder = validation_core.compile_model(ENCODER_PATH, "CPU")
validation_decoder = validation_core.compile_model(DECODER_PATH, "CPU")

ov_embeddings = validation_encoder({"input_image": encoder_input.numpy()})[validation_encoder.output("image_embeddings")]
ov_outputs = validation_decoder(
    {
        "image_embeddings": ov_embeddings,
        "point_coords": point_coords.numpy(),
        "point_labels": point_labels.numpy(),
        "mask_input": mask_input.numpy(),
        "has_mask_input": has_mask_input.numpy(),
        "orig_im_size": original_size_tensor.numpy(),
    }
)
ov_masks = ov_outputs[validation_decoder.output("masks")]
ov_scores = ov_outputs[validation_decoder.output("iou_predictions")]
ov_low_res = ov_outputs[validation_decoder.output("low_res_masks")]

report_difference("Image embeddings", image_embeddings.numpy(), ov_embeddings)
report_difference("Low-resolution masks", reference_low_res.numpy(), ov_low_res)
report_difference("IoU predictions", reference_scores.numpy(), ov_scores)
report_difference("Final mask logits", reference_masks.numpy(), ov_masks)

reference_binary = reference_masks.numpy() > sam.mask_threshold
ov_binary = ov_masks > sam.mask_threshold
intersection = np.logical_and(reference_binary, ov_binary).sum(axis=(-2, -1))
union = np.logical_or(reference_binary, ov_binary).sum(axis=(-2, -1))
mask_iou = intersection / np.maximum(union, 1)
print(f"Minimum final-mask IoU: {mask_iou.min():.6f}")

assert np.isfinite(ov_embeddings).all() and np.isfinite(ov_masks).all()
assert mask_iou.min() > 0.98
assert np.max(np.abs(reference_scores.numpy() - ov_scores)) < 0.02

## Compile the OpenVINO models
[back to top ⬆️](#Table-of-contents:)


In [ ]:
core = ov.Core()
compiled_encoder = core.compile_model(ENCODER_PATH, device.value)
compiled_decoder = core.compile_model(DECODER_PATH, device.value)

print(f"Device: {device.value}")
print("Encoder:", [(port.any_name, port.partial_shape) for port in compiled_encoder.inputs], "->", [port.any_name for port in compiled_encoder.outputs])
print("Decoder inputs:", [(port.any_name, port.partial_shape) for port in compiled_decoder.inputs])
print("Decoder outputs:", [port.any_name for port in compiled_decoder.outputs])

## Implement OpenVINO MobileSAM inference
[back to top ⬆️](#Table-of-contents:)

The predictor performs resize, normalization, and padding with NumPy and Pillow, so PyTorch is not required after conversion. Calling `set_image` caches the image embedding. Subsequent calls to `predict` execute only the prompt decoder.

In [10]:
PIXEL_MEAN = np.array([123.675, 116.28, 103.53], dtype=np.float32)
PIXEL_STD = np.array([58.395, 57.12, 57.375], dtype=np.float32)
IMAGE_SIZE = 1024


def get_preprocess_shape(height: int, width: int, target_length: int) -> tuple[int, int]:
    """Calculate a resize that preserves aspect ratio and fixes the longest side."""
    scale = target_length / max(height, width)
    return int(height * scale + 0.5), int(width * scale + 0.5)


def preprocess_image(input_image: np.ndarray, target_length: int = IMAGE_SIZE) -> np.ndarray:
    """Resize, normalize, and pad an RGB image for MobileSAM."""
    new_height, new_width = get_preprocess_shape(input_image.shape[0], input_image.shape[1], target_length)
    resized = np.asarray(Image.fromarray(input_image).resize((new_width, new_height), Image.Resampling.BILINEAR), dtype=np.float32)
    normalized = (resized - PIXEL_MEAN.reshape(1, 1, 3)) / PIXEL_STD.reshape(1, 1, 3)
    padded = np.zeros((target_length, target_length, 3), dtype=np.float32)
    padded[:new_height, :new_width] = normalized
    return np.ascontiguousarray(padded.transpose(2, 0, 1)[None])


class OpenVINOMobileSAMPredictor:
    """Run MobileSAM encoder and prompt decoder with OpenVINO."""

    def __init__(self, encoder: ov.CompiledModel, decoder: ov.CompiledModel, image_size: int = IMAGE_SIZE) -> None:
        self.encoder = encoder
        self.decoder = decoder
        self.image_size = image_size
        self.image: Optional[np.ndarray] = None
        self.embedding: Optional[np.ndarray] = None

    def set_image(self, image: np.ndarray) -> None:
        """Encode an RGB image and cache its embedding."""
        if image.ndim != 3 or image.shape[2] != 3:
            raise ValueError(f"Expected an HWC RGB image, received {image.shape}")
        self.image = image
        self.embedding = self.encoder({"input_image": preprocess_image(image, self.image_size)})[self.encoder.output("image_embeddings")]

    def _transform_coords(self, coordinates: np.ndarray) -> np.ndarray:
        """Transform XY coordinates from original pixels to encoder input pixels."""
        if self.image is None:
            raise RuntimeError("Call set_image before predict")
        height, width = self.image.shape[:2]
        new_height, new_width = get_preprocess_shape(height, width, self.image_size)
        transformed = coordinates.astype(np.float32, copy=True)
        transformed[..., 0] *= new_width / width
        transformed[..., 1] *= new_height / height
        return transformed

    def predict(
        self,
        point_coords: Optional[np.ndarray] = None,
        point_labels: Optional[np.ndarray] = None,
        box: Optional[np.ndarray] = None,
        multimask_output: bool = True,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Predict masks from positive/negative points, a box, or both."""
        if self.image is None or self.embedding is None:
            raise RuntimeError("Call set_image before predict")
        coordinate_parts: list[np.ndarray] = []
        label_parts: list[np.ndarray] = []
        if point_coords is not None:
            if point_labels is None:
                raise ValueError("point_labels must accompany point_coords")
            coordinate_parts.append(self._transform_coords(np.asarray(point_coords).reshape(-1, 2)))
            label_parts.append(np.asarray(point_labels, dtype=np.float32).reshape(-1))
        if box is not None:
            coordinate_parts.append(self._transform_coords(np.asarray(box).reshape(2, 2)))
            label_parts.append(np.array([2.0, 3.0], dtype=np.float32))
        if not coordinate_parts:
            raise ValueError("Provide at least one point or box prompt")

        coordinates = np.concatenate(coordinate_parts)[None].astype(np.float32)
        labels = np.concatenate(label_parts)[None].astype(np.float32)
        height, width = self.image.shape[:2]
        outputs = self.decoder(
            {
                "image_embeddings": self.embedding,
                "point_coords": coordinates,
                "point_labels": labels,
                "mask_input": np.zeros((1, 1, 256, 256), dtype=np.float32),
                "has_mask_input": np.zeros(1, dtype=np.float32),
                "orig_im_size": np.array([height, width], dtype=np.float32),
            }
        )
        masks = outputs[self.decoder.output("masks")][0]
        scores = outputs[self.decoder.output("iou_predictions")][0]
        low_res_masks = outputs[self.decoder.output("low_res_masks")][0]
        selection = slice(1, 4) if multimask_output else slice(int(np.argmax(scores)), int(np.argmax(scores)) + 1)
        return masks[selection] > 0.0, scores[selection], low_res_masks[selection]


predictor = OpenVINOMobileSAMPredictor(compiled_encoder, compiled_decoder)
predictor.set_image(image)

## Run inference on a sample image
[back to top ⬆️](#Table-of-contents:)

A label of `1` is a positive point and `0` is a negative point. A box is represented by its top-left and bottom-right XY coordinates.

In [ ]:
def show_mask(mask: np.ndarray, axis: plt.Axes, color: tuple[float, float, float, float] = (0.12, 0.56, 1.0, 0.55)) -> None:
    """Overlay a binary mask on a Matplotlib axis."""
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[mask] = color
    axis.imshow(overlay)


def show_points(coords: np.ndarray, labels: np.ndarray, axis: plt.Axes) -> None:
    """Draw positive and negative point prompts."""
    positive = coords[labels == 1]
    negative = coords[labels == 0]
    axis.scatter(positive[:, 0], positive[:, 1], color="lime", marker="*", s=250, edgecolor="white", linewidth=1.5)
    axis.scatter(negative[:, 0], negative[:, 1], color="red", marker="*", s=250, edgecolor="white", linewidth=1.5)


point_prompt = np.array([[550.0, 300.0], [420.0, 320.0]], dtype=np.float32)
point_prompt_labels = np.array([1, 0], dtype=np.float32)
point_masks, point_scores, _ = predictor.predict(point_coords=point_prompt, point_labels=point_prompt_labels)
point_best = int(np.argmax(point_scores))

box_prompt = np.array([435.0, 100.0, 650.0, 500.0], dtype=np.float32)
box_masks, box_scores, _ = predictor.predict(box=box_prompt)
box_best = int(np.argmax(box_scores))

figure, axes = plt.subplots(1, 2, figsize=(16, 7))
for axis in axes:
    axis.imshow(image)
    axis.axis("off")

show_mask(point_masks[point_best], axes[0])
show_points(point_prompt, point_prompt_labels, axes[0])
axes[0].set_title(f"Point prompts — predicted IoU {point_scores[point_best]:.3f}")

show_mask(box_masks[box_best], axes[1], color=(1.0, 0.45, 0.12, 0.5))
x0, y0, x1, y1 = box_prompt
axes[1].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, edgecolor="lime", facecolor="none", linewidth=2))
axes[1].set_title(f"Box prompt — predicted IoU {box_scores[box_best]:.3f}")
figure.savefig("file.png", bbox_inches="tight")
plt.show()

## Create an interactive segmentation demo
[back to top ⬆️](#Table-of-contents:)

Upload an image or use the `coco_bike.jpg` example, then click an object to segment it. The image embedding is cached by `set_image`; moving the prompt runs only the decoder. The demo uses Gradio's image component to provide click coordinates.

In [ ]:
demo_predictor = OpenVINOMobileSAMPredictor(compiled_encoder, compiled_decoder)


def cache_demo_image(input_image: np.ndarray) -> np.ndarray:
    """Cache the image embedding when the demo image changes."""
    demo_predictor.set_image(input_image.astype(np.uint8))
    return input_image


def segment_click(input_image: np.ndarray, event: gr.SelectData) -> np.ndarray:
    """Segment the object at a selected image coordinate."""
    if demo_predictor.image is None or demo_predictor.image.shape != input_image.shape:
        demo_predictor.set_image(input_image.astype(np.uint8))
    coordinates = np.array([[event.index[0], event.index[1]]], dtype=np.float32)
    masks, scores, _ = demo_predictor.predict(point_coords=coordinates, point_labels=np.array([1.0], dtype=np.float32))
    mask = masks[int(np.argmax(scores))]
    overlay = input_image.astype(np.float32).copy()
    color = np.array([30.0, 144.0, 255.0], dtype=np.float32)
    overlay[mask] = 0.55 * overlay[mask] + 0.45 * color
    cv2.drawMarker(overlay, tuple(coordinates[0].astype(int)), (0, 255, 0), cv2.MARKER_STAR, 22, 2)
    return overlay.astype(np.uint8)


with gr.Blocks() as demo:
    gr.Markdown("## MobileSAM with OpenVINO\nUpload an image, then click an object to segment it.")
    with gr.Row():
        demo_input = gr.Image(value=str(SAMPLE_PATH), label="Input — click an object", type="numpy")
        demo_output = gr.Image(label="Segmentation", type="numpy")
    demo_input.upload(cache_demo_image, inputs=demo_input, outputs=demo_input)
    demo_input.select(segment_click, inputs=demo_input, outputs=demo_output)

cache_demo_image(image)
# If launching remotely, specify server_name and server_port. See https://www.gradio.app/docs/gradio/blocks#launch
demo.launch(prevent_thread_lock=True)

## Measure model size and performance
[back to top ⬆️](#Table-of-contents:)

The following lightweight measurement runs two warm-up iterations followed by five measured iterations for each model. Progress messages identify the active stage. Results depend on the selected device, system load, and OpenVINO version; they are not fixed benchmark claims.

In [ ]:
encoder_benchmark_input = preprocess_image(image)
decoder_benchmark_input = {
    "image_embeddings": predictor.embedding,
    "point_coords": predictor._transform_coords(np.array([[550.0, 300.0]], dtype=np.float32))[None],
    "point_labels": np.array([[1.0]], dtype=np.float32),
    "mask_input": np.zeros((1, 1, 256, 256), dtype=np.float32),
    "has_mask_input": np.zeros(1, dtype=np.float32),
    "orig_im_size": np.array(image.shape[:2], dtype=np.float32),
}

warmup_iterations = 2
iterations = 5
print(f"Warming up the encoder and decoder ({warmup_iterations} iterations each)...", flush=True)
for _ in range(warmup_iterations):
    compiled_encoder({"input_image": encoder_benchmark_input})
    compiled_decoder(decoder_benchmark_input)

print(f"Measuring encoder latency ({iterations} iterations)...", flush=True)
start = time.perf_counter()
for _ in range(iterations):
    compiled_encoder({"input_image": encoder_benchmark_input})
encoder_latency = (time.perf_counter() - start) * 1000 / iterations

print(f"Measuring decoder latency ({iterations} iterations)...", flush=True)
start = time.perf_counter()
for _ in range(iterations):
    compiled_decoder(decoder_benchmark_input)
decoder_latency = (time.perf_counter() - start) * 1000 / iterations


def ir_size_megabytes(xml_path: Path) -> float:
    """Return the combined XML and BIN artifact size in MiB."""
    return (xml_path.stat().st_size + xml_path.with_suffix(".bin").stat().st_size) / (1024**2)


print(f"Device-dependent encoder latency: {encoder_latency:.2f} ms")
print(f"Device-dependent decoder latency with cached embedding: {decoder_latency:.2f} ms")
print(f"Approximate first-prompt latency: {encoder_latency + decoder_latency:.2f} ms")
print(f"Encoder IR size: {ir_size_megabytes(ENCODER_PATH):.2f} MiB")
print(f"Decoder IR size: {ir_size_megabytes(DECODER_PATH):.2f} MiB")

## Verify notebook reproducibility
[back to top ⬆️](#Table-of-contents:)

These executable checks cover downloaded assets, IR serialization, dynamic prompt support, tensor shapes, finite outputs, and nonempty masks. Repository CI additionally runs notebook formatting, metadata, telemetry, and clean-kernel execution checks.

In [ ]:
required_files = [SOURCE_DIR / "mobile_sam" / "__init__.py", CHECKPOINT_PATH, SAMPLE_PATH, ENCODER_PATH, DECODER_PATH]
assert all(path.exists() and path.stat().st_size > 0 for path in required_files)
assert predictor.embedding is not None and predictor.embedding.shape == (1, 256, 64, 64)
assert point_masks.shape[0] == 3 and point_masks.shape[1:] == image.shape[:2]
assert box_masks.shape[0] == 3 and box_masks.shape[1:] == image.shape[:2]
assert point_masks.any() and box_masks.any()
assert np.isfinite(point_scores).all() and np.isfinite(box_scores).all()
assert compiled_decoder.input("point_coords").partial_shape[1].is_dynamic
print("All reproducibility checks passed.")

### Optional cleanup
[back to top ⬆️](#Table-of-contents:)

Uncomment and run the following cell to remove downloaded source, model artifacts, and sample data.

In [ ]:
# import shutil

# shutil.rmtree(SOURCE_DIR, ignore_errors=True)
# shutil.rmtree(MODEL_DIR, ignore_errors=True)
# shutil.rmtree(DATA_DIR, ignore_errors=True)
# UTILS_PATH.unlink(missing_ok=True)